In [0]:
# SparkSession.builder: Used to configure a Spark application.
# getOrCreate():
#           Returns the existing Spark session if one already exists.
#           Otherwise, creates a new Spark session.
from pyspark.sql import SparkSession
from delta.tables import DeltaTable

spark = SparkSession.builder.getOrCreate()

#### Creating the delta table from dataframe

In [0]:
# Existing target table data
target_data = [
    (1, "John", 4500),
    (2, "Alice", 6000),
    (3, "Bob", 5500)]

target_df = spark.createDataFrame(
    target_data,
    ["id", "name", "salary"])

# Save as Delta table
target_df.write.format("delta").mode("overwrite").saveAsTable("employee")

#### CDC (Change Data Capture) Using Delta Lake MERGE 

In [0]:
from delta.tables import DeltaTable

# Create a Delta Object on the table 'employee' to perform Merge. 
target = DeltaTable.forName(spark, "employee")
# Incoming CDC data
source_data = [
    (1, "John", 5000),   # Updated salary
    (2, "Alice", 6000),  # No change
    (4, "Tom", 7000)     # New record
]

source_df = spark.createDataFrame(source_data, ["id", "name", "salary"])

# Create a Delta Object on the table 'employee' to perform Merge. 
target = DeltaTable.forName(spark, "employee")

target.alias("t").merge(source_df.alias("s"),
                        "t.id = s.id")           \
                .whenMatchedUpdate(set={"name": "s.name",
                                        "salary": "s.salary"})  \
                .whenNotMatchedInsert(values={"id": "s.id",
                                              "name": "s.name",
                                              "salary": "s.salary"})  \
                .execute()

# View Result
spark.table("employee").orderBy("id").show()

+---+-----+------+
| id| name|salary|
+---+-----+------+
|  1| John|  5000|
|  2|Alice|  6000|
|  3|  Bob|  5500|
|  4|  Tom|  7000|
+---+-----+------+



#### Finding top 2 salary records for each department using **Window** function

In [0]:
data = [
    (101, "John",    "IT",        75000, "2026-01-15"),
    (102, "Mary",    "HR",        65000, "2025-11-20"),
    (103, "David",   "Finance",   84000, "2024-09-10"),
    (104, "Sarah",   "IT",        72000, "2026-03-05"),
    (105, "Michael", "Sales",     68000, "2025-07-18"),
    (106, "Emma",    "Marketing", 70000, "2024-12-12"),
    (107, "James",   "IT",        85000, "2026-04-22"),
    (108, "Sophia",  "Finance",   78000, "2025-01-30"),
    (109, "Daniel",  "Sales",     62000, "2026-02-14"),
    (110, "Olivia",  "HR",        67000, "2024-10-08"),
    (111, "Liam",    "IT",        88000, "2025-08-17"),
    (112, "Ava",     "Marketing", 69000, "2026-05-25"),
    (113, "Noah",    "Finance",   81000, "2025-06-11"),
    (114, "Isabella","Sales",     68000, "2026-07-01"),
    (115, "Mason",   "IT",        90000, "2024-11-19"),
    (116, "Mia",     "HR",        71000, "2025-03-22"),
    (117, "Ethan",   "Marketing", 73000, "2026-06-09"),
    (118, "Charlotte","Finance",  84000, "2024-08-27"),
    (119, "Logan",   "Sales",     66000, "2025-12-03"),
    (120, "Amelia",  "IT",        90000, "2026-08-15")
]

columns = ["emp_id", "emp_name", "department", "salary", "join_date"]

df=spark.createDataFrame(data,columns)
df.show(5)

+------+--------+----------+------+----------+
|emp_id|emp_name|department|salary| join_date|
+------+--------+----------+------+----------+
|   101|    John|        IT| 75000|2026-01-15|
|   102|    Mary|        HR| 65000|2025-11-20|
|   103|   David|   Finance| 84000|2024-09-10|
|   104|   Sarah|        IT| 72000|2026-03-05|
|   105| Michael|     Sales| 68000|2025-07-18|
+------+--------+----------+------+----------+
only showing top 5 rows


In [0]:
from pyspark.sql import Window
from pyspark.sql import functions as F

window=Window.partitionBy(F.col('department')).orderBy(F.col('salary').desc())

print('Using Row Number')
df_dedup=df.withColumn('rownumber',F.row_number().over(window)).filter(F.col('rownumber') <= 2 )
df_dedup.orderBy('department').show()

print('Using Rank')
df_dedup=df.withColumn('rownumber',F.rank().over(window)).filter(F.col('rownumber') <= 2)
df_dedup.orderBy('department').show()

print('Using dense_rank')
df_dedup=df.withColumn('rownumber',F.dense_rank().over(window)).filter(F.col('rownumber') <= 2)
df_dedup.orderBy('department').show()

Using Row Number
+------+---------+----------+------+----------+---------+
|emp_id| emp_name|department|salary| join_date|rownumber|
+------+---------+----------+------+----------+---------+
|   103|    David|   Finance| 84000|2024-09-10|        1|
|   118|Charlotte|   Finance| 84000|2024-08-27|        2|
|   116|      Mia|        HR| 71000|2025-03-22|        1|
|   110|   Olivia|        HR| 67000|2024-10-08|        2|
|   115|    Mason|        IT| 90000|2024-11-19|        1|
|   120|   Amelia|        IT| 90000|2026-08-15|        2|
|   117|    Ethan| Marketing| 73000|2026-06-09|        1|
|   106|     Emma| Marketing| 70000|2024-12-12|        2|
|   105|  Michael|     Sales| 68000|2025-07-18|        1|
|   114| Isabella|     Sales| 68000|2026-07-01|        2|
+------+---------+----------+------+----------+---------+

Using Rank
+------+---------+----------+------+----------+---------+
|emp_id| emp_name|department|salary| join_date|rownumber|
+------+---------+----------+------+-------

#### Creating **schema** manually for dataframe

In [0]:
data = ([(101, 'Sachin'),(102,'Dravid'),(103,'Ganguly'),(104,'Kumble'),(105,'Sehwag'),(106,'Kohli'),(107,'Dhoni'),(108,'Yuvraj'),(109,'Raina'),(110,'Zaheer')])

from pyspark.sql.types import StructType, StructField, StringType, IntegerType

schema = StructType([StructField('id',IntegerType()),
                    StructField('name',StringType())])

df=spark.createDataFrame(data,schema)
df.show()


+---+-------+
| id|   name|
+---+-------+
|101| Sachin|
|102| Dravid|
|103|Ganguly|
|104| Kumble|
|105| Sehwag|
|106|  Kohli|
|107|  Dhoni|
|108| Yuvraj|
|109|  Raina|
|110| Zaheer|
+---+-------+



#### Creating a table and viewing the data from table

In [0]:
# Creating a table 'players' from the dataframe df
df.write.mode('overwrite').option("overwriteSchema", "true").saveAsTable('players')

# View the table data using spark.sql
spark.sql("SELECT * FROM players order by id desc").show()

+---+-------+
| id|   name|
+---+-------+
|110| Zaheer|
|109|  Raina|
|108| Yuvraj|
|107|  Dhoni|
|106|  Kohli|
|105| Sehwag|
|104| Kumble|
|103|Ganguly|
|102| Dravid|
|101| Sachin|
+---+-------+



#### Reading .csv file from source path and writing to Bronze table with metadata columns added. 

In [0]:
from pyspark.sql import functions as F

# Configuration
SOURCE_NAME = "players_csv"
SOURCE_PATH = "/Volumes/workspace/default/landing_volumne/data.csv"

BRONZE_TABLE = "workspace.default.customer_bronze"

# Read the first CSV file
data_df = spark.read  \
    .option("header", True)    \
    .option("inferSchema", True)    \
    .csv(SOURCE_PATH)

# Add metadata columns
bronze_df = data_df   \
    .withColumn("_source_file", F.col("_metadata.file_path"))   \
    .withColumn("_source_name", F.lit(SOURCE_NAME))    \
    .withColumn("_ingestion_timestamp", F.current_timestamp())

bronze_df.show(5)

# Write to the Bronze table
bronze_df.write   \
             .format("delta")    \
             .mode("overwrite")     \
             .option("mergeSchema", "true")    \
             .saveAsTable(BRONZE_TABLE)

+---+-------+--------------------+------------+--------------------+
| id|   name|        _source_file|_source_name|_ingestion_timestamp|
+---+-------+--------------------+------------+--------------------+
|101| Sachin|dbfs:/Volumes/wor...| players_csv|2026-09-20 04:51:...|
|102| Dravid|dbfs:/Volumes/wor...| players_csv|2026-09-20 04:51:...|
|103|Ganguly|dbfs:/Volumes/wor...| players_csv|2026-09-20 04:51:...|
|104| Kumble|dbfs:/Volumes/wor...| players_csv|2026-09-20 04:51:...|
|105| Sehwag|dbfs:/Volumes/wor...| players_csv|2026-09-20 04:51:...|
+---+-------+--------------------+------------+--------------------+
only showing top 5 rows


#### Reading the data from Bronze Table using spark and sql

In [0]:
# Read the Bronze table along with its schema
spark.table(BRONZE_TABLE).printSchema()
spark.table(BRONZE_TABLE).show(5) 

root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _source_name: string (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- country: string (nullable = true)

+---+-------+--------------------+------------+--------------------+-------+
| id|   name|        _source_file|_source_name|_ingestion_timestamp|country|
+---+-------+--------------------+------------+--------------------+-------+
|101| Sachin|dbfs:/Volumes/wor...| players_csv|2026-09-20 04:51:...|   NULL|
|102| Dravid|dbfs:/Volumes/wor...| players_csv|2026-09-20 04:51:...|   NULL|
|103|Ganguly|dbfs:/Volumes/wor...| players_csv|2026-09-20 04:51:...|   NULL|
|104| Kumble|dbfs:/Volumes/wor...| players_csv|2026-09-20 04:51:...|   NULL|
|105| Sehwag|dbfs:/Volumes/wor...| players_csv|2026-09-20 04:51:...|   NULL|
+---+-------+--------------------+------------+--------------------+-------+
only showing top 5 rows


In [0]:
%sql
SELECT * FROM workspace.default.customer_bronze LIMIT 5;

id,name,_source_file,_source_name,_ingestion_timestamp,country
101,Sachin,dbfs:/Volumes/workspace/default/landing_volumne/data.csv,players_csv,2026-09-20T04:51:09.775Z,null
102,Dravid,dbfs:/Volumes/workspace/default/landing_volumne/data.csv,players_csv,2026-09-20T04:51:09.775Z,null
103,Ganguly,dbfs:/Volumes/workspace/default/landing_volumne/data.csv,players_csv,2026-09-20T04:51:09.775Z,null
104,Kumble,dbfs:/Volumes/workspace/default/landing_volumne/data.csv,players_csv,2026-09-20T04:51:09.775Z,null
105,Sehwag,dbfs:/Volumes/workspace/default/landing_volumne/data.csv,players_csv,2026-09-20T04:51:09.775Z,null


#### Handling Schema Drift

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

def detect_schema_drift(new_schema):

    expected_schema = StructType([
        StructField("id", IntegerType(), True),
        StructField("name", StringType(), True)
    ])

    # Convert schemas into dictionaries: column -> datatype
    expected_dt = {
        field.name: field.dataType.simpleString()
        for field in expected_schema.fields
    }

    new_dt = {
        field.name: field.dataType.simpleString()
        for field in new_schema.fields
    }

    # Compare column names
    expected_columns = set(expected_schema.fieldNames())
    actual_columns = set(new_schema.fieldNames())

    missing_columns = expected_columns - actual_columns
    new_columns = actual_columns - expected_columns

    # Check datatype changes for common columns
    common_columns = expected_columns & actual_columns
    
    datatype_changes = {
        column: { "expected": expected_dt[column],
                  "actual": new_dt[column]}
        for column in common_columns
        if expected_dt[column] != new_dt[column]}

    # Print only when drift is detected
    if missing_columns or new_columns or datatype_changes:
        print("Schema drift detected!")

        if missing_columns:
            print("Missing columns:", missing_columns)

        if new_columns:
            print("New columns:", new_columns)

        if datatype_changes:
            print("Datatype changes:")

            for column, datatype in datatype_changes.items():
                print(
                    f"Column '{column}': "
                    f"expected {datatype['expected']}, "
                    f"but found {datatype['actual']}"
                )
    else:
        print("No schema drift detected.")

print ("Reading the first CSV file...")
# Read the first CSV file
data_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("/Volumes/workspace/default/landing_volumne/data.csv")
)

# Pass data_df.schema, not data_df.printSchema()
detect_schema_drift(data_df.schema)

print ("\nReading the second CSV file...")
# Read the second CSV file
new_data_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("/Volumes/workspace/default/landing_volumne/new_data.csv")
)

# Pass new_data_df.schema
detect_schema_drift(new_data_df.schema)


Reading the first CSV file...
No schema drift detected.

Reading the second CSV file...
Schema drift detected!
New columns: {'country'}


In [0]:
# Reading using the SOURCE_PATH details. 


from pyspark.sql import functions as F

# Configuration
SOURCE_NAME = "players_csv"
SOURCE_PATH = "/Volumes/workspace/default/landing_volumne/new_data.csv"
BRONZE_TABLE = "workspace.default.customer_bronze"

# Read the next new CSV file
data_df = spark.read  \
    .option("header", True)    \
    .option("inferSchema", True)    \
    .csv(SOURCE_PATH)

bronze_df = data_df   \
    .withColumn("_source_file", F.col("_metadata.file_path"))   \
    .withColumn("_source_name", F.lit(SOURCE_NAME))    \
    .withColumn("_ingestion_timestamp", F.current_timestamp())

bronze_df.write   \
             .format("delta")    \
             .mode("append")     \
             .option("mergeSchema", "true")    \
             .option("overwriteSchema", "true")    \
             .saveAsTable(BRONZE_TABLE)

#### Handling corrupt records

##### Ignoring new column

In [0]:
# This code will not give any error. read.csv() will just read the 2 columns from the input and ignore other new columns.

SOURCE_PATH = "/Volumes/workspace/default/landing_volumne/new_data.csv"

expected_schema = StructType([
        StructField("id", IntegerType(), True),
        StructField("name", StringType(), True)
    ])

data_m_df = spark.read  \
    .schema(expected_schema)   \
    .option('Header', True)    \
    .csv(SOURCE_PATH)

data_m_df.show(6)

+---+-------+
| id|   name|
+---+-------+
|101| Sachin|
|102| Dravid|
|103|Ganguly|
|104| Kumble|
|105| Sehwag|
|106|  Kohli|
+---+-------+
only showing top 6 rows


##### Missing column in new file

In [0]:
# This code will not give any error. read.csv() will just read the 1 column data from the input and since 2nd is missing that column data will be marked as NULL. 

SOURCE_PATH = "/Volumes/workspace/default/landing_volumne/missing_col.csv"

expected_schema = StructType([
        StructField("id", IntegerType(), True),
        StructField("name", StringType(), True)
    ])

missing_col_df = spark.read  \
    .schema(expected_schema)   \
    .option('Header', True)    \
    .csv(SOURCE_PATH)

missing_col_df.show(6)

+---+----+
| id|name|
+---+----+
|123|NULL|
|234|NULL|
|345|NULL|
|567|NULL|
|678|NULL|
|822|NULL|
+---+----+
only showing top 6 rows


### Production version of detecting corrupt record

##### **PERMISSIVE**

In [0]:
# This code will not give any error. PERMISSIVE will ignore the corrupt records making corrupt record as NULL

SOURCE_PATH = "/Volumes/workspace/default/landing_volumne/datatype_error_data.csv"

# Defining the expected schema
expected_schema = StructType([
        StructField("id", IntegerType(), True),
        StructField("name", StringType(), True),
        StructField("country", StringType(), True)
    ])

# Reading the file from source path with PERMISSIVE mode. 
data_e_df = (spark.read
            .schema(expected_schema)
            .option("header", "true")
            .option("mode", "PERMISSIVE")
            .csv(SOURCE_PATH))

# Adding RejectReason column
data_e_df = data_e_df.withColumn("RejectReason",F.concat_ws(", ",
    F.when(F.col("id").isNull(),"ID is invalid"),
    F.when((F.col("name").isNull()) | (F.trim(F.col("name")) == ""), "Name is Missing"),
    F.when((F.col("country").isNull()) | (F.trim(F.col("country")) == ""), "Country is Missing")))

# Separate the valid and invalid records before writting into the target
valid_recs = data_e_df.filter(F.col('RejectReason') == '').drop('RejectReason')
invalid_recs = data_e_df.filter(F.col('RejectReason') != '')

print("Valid records are:")
valid_recs.show(5)
print("Invalid records are:")
invalid_recs.show(5)  

Valid records are:
+---+-------+-------+
| id|   name|country|
+---+-------+-------+
|101| Sachin|  India|
|102| Dravid|  India|
|103|Ganguly|  India|
|104| Kumble|  India|
|106|  Kohli|  India|
+---+-------+-------+
only showing top 5 rows
Invalid records are:
+----+------+-------+--------------------+
|  id|  name|country|        RejectReason|
+----+------+-------+--------------------+
|NULL|Sehwag|  India|       ID is invalid|
| 107|      |  India|     Name is Missing|
| 110|Zaheer|       |  Country is Missing|
|NULL|  NULL|   NULL|ID is invalid, Na...|
+----+------+-------+--------------------+



##### **DROPMALFORMED**

In [0]:
# This code will not give any error. DROPMALFORMED will DROP the corrupt record(DataType issue recs). DATA Loss. 
# DROPMALFORMED doesn't Drop rows that violate business mandatory-field rules (recs having spaces in string)

SOURCE_PATH = "/Volumes/workspace/default/landing_volumne/datatype_error_data1.csv"

expected_schema = StructType([
        StructField("id", IntegerType(), True),
        StructField("name", StringType(), True), 
        StructField("country", StringType(), True)
    ])

data_e_df = spark.read  \
    .schema(expected_schema)   \
    .option('header', 'true')    \
    .option("ignoreLeadingWhiteSpace", "true") \
    .option("ignoreTrailingWhiteSpace", "true") \
    .option("mode", "DROPMALFORMED")    \
    .csv(SOURCE_PATH)

# Validation
validation_condition = (F.col('id').isNotNull()        &
                        F.col("name").isNotNull()      &
                        (F.trim(F.col("name")) != '')  &
                        F.col("country").isNotNull()   &
                        (F.trim(F.col("country")) != ''))

# Separate the valid and invalid records before writting into the target
valid_recs = data_e_df.filter(validation_condition)
invalid_recs = data_e_df.filter(~validation_condition)

print("Valid records are:")
valid_recs.show(5)
print("Invalid records are:")
invalid_recs.show(5) 

Valid records are:
+---+-------+-------+
| id|   name|country|
+---+-------+-------+
|101| Sachin|  India|
|102| Dravid|  India|
|103|Ganguly|  India|
|104| Kumble|  India|
|106|  Kohli|  India|
+---+-------+-------+
only showing top 5 rows
Invalid records are:
+---+------+-------+
| id|  name|country|
+---+------+-------+
|107|  NULL|  India|
|110|Zaheer|   NULL|
+---+------+-------+



Below is a **production-oriented pattern** that handles **schema drift, malformed CSV records, datatype errors, null/blank values, audit information, and quarantine records** without silently dropping data.

**Recommended design**: First validate the CSV header against the expected schema. Only when the file structure is valid should you apply the typed schema and perform record-level validation.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, StringType
from datetime import datetime

# ============================================================
# 1. CONFIGURATION
# ============================================================

SOURCE_NAME = "customer_csv"

SOURCE_PATH = ("/Volumes/workspace/default/landing_volumne/final_data.csv")

EXPECTED_SCHEMA = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("country", StringType(), True)
])

CORRUPT_COLUMN = "_corrupt_record"

# Schema drift policies:
# "FAIL"       -> Stop processing when drift is detected
# "QUARANTINE" -> Return schema drift details without processing records
SCHEMA_DRIFT_POLICY = "QUARANTINE"
EXPECTED_COLUMNS = [field.name for field in EXPECTED_SCHEMA.fields]

# ============================================================
# 2. READ ONLY THE CSV HEADER FOR SCHEMA DRIFT VALIDATION
# ============================================================
header_df = (
    spark.read
         .format("csv")
         .option("header", "true")
         .option("inferSchema", "false")
         .option("ignoreLeadingWhiteSpace", "true")
         .option("ignoreTrailingWhiteSpace", "true")
         .load(SOURCE_PATH)
)
actual_columns = header_df.columns
print(f"Expected columns : {EXPECTED_COLUMNS}")
print(f"Actual columns   : {actual_columns}")
# ============================================================
# 3. DETECT DIFFERENT TYPES OF SCHEMA DRIFT
# ============================================================
missing_columns = [column for column in EXPECTED_COLUMNS if column not in actual_columns]
new_columns = [column for column in actual_columns if column not in EXPECTED_COLUMNS]
common_actual_columns = [column for column in actual_columns if column in EXPECTED_COLUMNS]
common_expected_columns = [column for column in EXPECTED_COLUMNS if column in actual_columns]
column_order_changed = (common_actual_columns != common_expected_columns)
duplicate_columns = list({column for column in actual_columns if actual_columns.count(column) > 1 })

schema_drift_detected = (len(missing_columns) > 0 or len(new_columns) > 0 or 
                         column_order_changed or len(duplicate_columns) > 0)
# ============================================================
# 4. CREATE SCHEMA DRIFT AUDIT DATAFRAME
# ============================================================
schema_audit_data = [(SOURCE_NAME,
                      SOURCE_PATH,
                      str(EXPECTED_COLUMNS),
                      str(actual_columns),
                      str(missing_columns),
                      str(new_columns),
                      str(duplicate_columns),
                      column_order_changed,
                      schema_drift_detected,
                      datetime.now())]

schema_audit_schema = StructType([
    StructField("source_name", StringType(), False),
    StructField("source_path", StringType(), False),
    StructField("expected_columns", StringType(), False),
    StructField("actual_columns", StringType(), False),
    StructField("missing_columns", StringType(), False),
    StructField("new_columns", StringType(), False),
    StructField("duplicate_columns", StringType(), False),
    StructField("column_order_changed", StringType(), True),
    StructField("schema_drift_detected", StringType(), True),
    StructField("audit_timestamp", StringType(), True)
])

schema_audit_df = spark.createDataFrame(
    schema_audit_data,
    [
        "source_name",
        "source_path",
        "expected_columns",
        "actual_columns",
        "missing_columns",
        "new_columns",
        "duplicate_columns",
        "column_order_changed",
        "schema_drift_detected",
        "audit_timestamp"
    ]
)

schema_audit_df.show(truncate=False)

#Stop or quarantine the file when schema drift exist
if schema_drift_detected:

    print("Schema drift detected.")

    if missing_columns:
        print(f"Missing columns    : {missing_columns}")
    if new_columns:
        print(f"New columns        : {new_columns}")
    if duplicate_columns:
        print(f"Duplicate columns  : {duplicate_columns}")
    if column_order_changed:
        print("Column order has changed.")
    if SCHEMA_DRIFT_POLICY == "FAIL":
        raise ValueError(
            f"Schema validation failed for {SOURCE_PATH}. "
            f"Missing columns: {missing_columns}, "
            f"New columns: {new_columns}, "
            f"Duplicate columns: {duplicate_columns}, "
            f"Column order changed: {column_order_changed}"
        )
    print("The complete source file must be moved to schema quarantine.")
else:
    print("Schema validation successful. Starting record validation.")

if not schema_drift_detected:
    # Add corrupt-record column to the expected schema
    ingestion_schema = StructType(EXPECTED_SCHEMA.fields + [StructField(CORRUPT_COLUMN, StringType(), True)])

    data_e_df = (spark.read
             .format("csv")
             .schema(ingestion_schema)
             .option("header", "true")
             .option("mode", "PERMISSIVE")
             .option("columnNameOfCorruptRecord",CORRUPT_COLUMN)
             .option("ignoreLeadingWhiteSpace", "true")
             .option("ignoreTrailingWhiteSpace", "true")
             .option("quote", '"')
             .option("escape", '"')
             .load(SOURCE_PATH)
             .withColumn("source_file",F.col("_metadata.file_path"))
             .withColumn("ingestion_timestamp",F.current_timestamp()))

    source_record_count = data_e_df.count()
    print(f"Source record count: {source_record_count}")

if not schema_drift_detected:
    validated_df = (data_e_df
        # Normalize string columns
        .withColumn("name", F.when(F.col("name").isNotNull(),F.trim(F.col("name"))))
        .withColumn("country",F.when(F.col("country").isNotNull(),F.trim(F.col("country"))))
        # Create an array containing every rejection reason
        .withColumn("reject_reasons_array",F.array(
                F.when( F.col(CORRUPT_COLUMN).isNotNull(),F.lit("Malformed CSV or datatype parsing error")),
                F.when(F.col("id").isNull(),F.lit("ID is null or invalid")),
                F.when(F.col("name").isNull(),F.lit("Name is null")),
                F.when(F.trim(F.col("name")) == "",F.lit("Name is blank")),
                F.when(F.col("country").isNull(), F.lit("Country is null")),
                F.when(F.trim(F.col("country")) == "", F.lit("Country is blank"))))
        # Remove null elements from the rejection-reason array
        .withColumn("reject_reasons_array",F.filter(F.col("reject_reasons_array"),
                lambda reason: reason.isNotNull()))
        # Convert all rejection reasons into one readable string
        .withColumn("reject_reason", F.concat_ws( "; ",F.col("reject_reasons_array")))
        # Add overall validation status
        .withColumn( "record_status",
            F.when(F.size(F.col("reject_reasons_array")) == 0,F.lit("VALID")
            ).otherwise(F.lit("INVALID")))
        .drop("reject_reasons_array"))
    
#Separate valid and invalid records
if not schema_drift_detected:
    valid_recs = (validated_df.filter(F.col("record_status") == "VALID")
        .drop(CORRUPT_COLUMN,"reject_reason","record_status"))

    invalid_recs = (validated_df.filter(F.col("record_status") == "INVALID"))

    valid_count = valid_recs.count()
    invalid_count = invalid_recs.count()

    print(f"Valid record count   : {valid_count}")
    print(f"Invalid record count : {invalid_count}") 

    print("Valid records:")
    valid_recs.show(truncate=False)

    print("Invalid records:")
    invalid_recs.show(truncate=False)

# write to Delta tables
if not schema_drift_detected:
    valid_recs.write \
        .format("delta") \
        .mode("append") \
        .option("mergeSchema", "false") \
        .saveAsTable("workspace.default.customer_silver")

    invalid_recs.write \
        .format("delta") \
        .mode("append") \
        .option("mergeSchema", "true") \
        .saveAsTable("workspace.default.customer_quarantine" )

    schema_audit_df.write \
        .format("delta") \
        .mode("append") \
        .option("mergeSchema", "true") \
        .saveAsTable("workspace.default.schema_drift_audit")

Expected columns : ['id', 'name', 'country']
Actual columns   : ['id', 'name', 'country']
+------------+---------------------------------------------------------+-------------------------+-------------------------+---------------+-----------+-----------------+--------------------+---------------------+--------------------------+
|source_name |source_path                                              |expected_columns         |actual_columns           |missing_columns|new_columns|duplicate_columns|column_order_changed|schema_drift_detected|audit_timestamp           |
+------------+---------------------------------------------------------+-------------------------+-------------------------+---------------+-----------+-----------------+--------------------+---------------------+--------------------------+
|customer_csv|/Volumes/workspace/default/landing_volumne/final_data.csv|['id', 'name', 'country']|['id', 'name', 'country']|[]             |[]         |[]               |false            

### Corrupted JSON file with PERMISSIVE mode. Input file is as below
''' {"id":101,"name":"Sachin"} 
{"id":102,"name":"Dravid"}
{"id":103,"name":"Ganguly"
{"id":104,"name":"Kumble"} '''

In [0]:
from pyspark.sql.types import *

schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    # Must be included in the user-defined schema
    StructField("_corrupt_record", StringType(), True)
])

df = (
    spark.read
    .schema(schema)
    .option("mode", "PERMISSIVE")
    .option("columnNameOfCorruptRecord", "_corrupt_record")
    .json("/Volumes/workspace/default/landing_volumne/corrupted_json.json")
)
df.show(truncate=False)

# In some Spark or Databricks versions, directly querying only _corrupt_record from a raw JSON source can produce # an error. It is safer to materialize or cache the DataFrame first:
# use df.cache() and then use valid_df = df.filter(F.col("_corrupt_record").isNull()

+----+------+--------------------------+
|id  |name  |_corrupt_record           |
+----+------+--------------------------+
|101 |Sachin|NULL                      |
|102 |Dravid|NULL                      |
|NULL|NULL  |{"id":103,"name":"Ganguly"|
|104 |Kumble|NULL                      |
+----+------+--------------------------+

